In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import h5py
import os
import pandas as pd
from pathlib import Path

In [ ]:
import sys
from omegaconf import OmegaConf

project_root = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from simulator import smFRET_simulator_wrapper_from_config, binding_simulator_wrapper_from_config
from priors import generate_fret_prior_from_config
from utils import eval_free_energy_from_config, eval_force_from_config

In [ ]:
ACCEPTOR_ID = 1
DONOR_ID = 2


def get_trajectory_data(traj_file):
    """Load a single trajectory from an HDF5 file.
    
    Returns
    -------
    duration_ms : float
        Total trajectory duration in milliseconds.
    interphoton_times : np.ndarray
        Inter-photon arrival times in MacroTimeUnits.
    macro_time_unit_ps : float
        Conversion factor: MacroTimeUnits -> picoseconds.
    detector : np.ndarray
        Detector IDs (1=acceptor, 2=donor).
    """
    with h5py.File(traj_file, "r") as f:
        detector = f["Detectors"][:]
        interphoton_ds = f["InterPhotonTimes"]
        interphoton_times = interphoton_ds[:]
        macro_time_unit_ps = interphoton_ds.attrs["MacroTimeUnitInPicoSeconds"]

        total_macrotime_units = int(interphoton_times.sum(dtype="uint64"))
        total_duration_ps = total_macrotime_units * int(macro_time_unit_ps)
        total_duration_s = total_duration_ps * 1e-12
        total_duration_ms = total_duration_s * 1e3

    return total_duration_ms, interphoton_times, macro_time_unit_ps, detector


def photons_to_timeseries(interphoton_times, macro_time_unit_ps, detector, bin_width_ms=0.5):
    """Bin photon arrivals into a donor/acceptor timeseries.
    
    Returns
    -------
    bin_centers_ms : np.ndarray
        Bin center times in milliseconds.
    donor_counts : np.ndarray
        Donor photon counts per bin.
    acceptor_counts : np.ndarray
        Acceptor photon counts per bin.
    """
    bin_width_ps = np.int64(bin_width_ms * 1e9)  # ms -> ps
    macro_time_unit_ps = np.int64(macro_time_unit_ps)

    interphoton_times_ps = interphoton_times.astype(np.int64) * macro_time_unit_ps
    photon_times_ps = np.cumsum(interphoton_times_ps, dtype=np.int64)

    total_time_ps = photon_times_ps[-1]
    n_bins = max(1, int(np.ceil(total_time_ps / bin_width_ps)))
    bin_edges = np.linspace(0, total_time_ps, n_bins + 1)

    donor_counts, _ = np.histogram(photon_times_ps[detector == DONOR_ID], bins=bin_edges)
    acceptor_counts, _ = np.histogram(photon_times_ps[detector == ACCEPTOR_ID], bins=bin_edges)

    bin_centers_ms = (bin_edges[:-1] + bin_edges[1:]) / 2 * 1e-9  # ps -> ms
    return bin_centers_ms, donor_counts, acceptor_counts


def bin_along_time(x, n_bins=100):
    """Bin a 1D array into n_bins by summing."""
    n = len(x)
    n_per_bin = int(n / n_bins)
    x_binned = np.zeros(n_bins)
    for i in range(n_bins):
        x_binned[i] = np.sum(x[i * n_per_bin:(i + 1) * n_per_bin])
    return x_binned


def rebin_simulated_trace(trace, sim_bins_per_vis_bin):
    """Rebin a simulated trace from fine sim bins to coarser visualization bins.
    
    Parameters
    ----------
    trace : np.ndarray, shape (n_sim_bins, 2)
        Simulated trace with [donor, acceptor] columns at SIM_BIN_WIDTH_MS resolution.
    sim_bins_per_vis_bin : int
        Number of simulator bins per visualization bin.
    
    Returns
    -------
    rebinned : np.ndarray, shape (n_vis_bins, 2)
        Rebinned trace at BIN_WIDTH_MS resolution.
    """
    n = len(trace)
    n_vis_bins = n // sim_bins_per_vis_bin
    if n_vis_bins == 0:
        return trace  # trace shorter than one vis bin
    trimmed = trace[:n_vis_bins * sim_bins_per_vis_bin]
    rebinned = trimmed.reshape(n_vis_bins, sim_bins_per_vis_bin, 2).sum(axis=1)
    return rebinned

# Section 1: Load and Explore Experimental Data

Data from **Sturzenegger et al. (2018)** *Nature Communications* 9, 4708 — transition path measurements of ACTR/NCBD coupled folding and binding.

Each HDF5 file contains one transition path event with photon-by-photon records:
- `InterPhotonTimes` — time between consecutive photons (in MacroTimeUnits)
- `Detectors` — 1 = acceptor, 2 = donor

There are 12 experimental conditions (varying salt, glycerol, dye pairs). Select one below.

In [ ]:
DATA_ROOT = Path("/home/dingeldein/Desktop/fret_sbi/data_ben_schuler/Sturzenegger_transition_path_data")
CSV_PATH = Path("/home/dingeldein/Desktop/fret_sbi/data_ben_schuler/trajectory_durations_sturzenegger_2018.csv")

# List available conditions
conditions = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
for i, cond in enumerate(conditions):
    n_files = len(list((DATA_ROOT / cond).glob("*.h5")))
    print(f"  [{i:2d}] {cond}  ({n_files} trajectories)")

# === SELECT CONDITION HERE ===
CONDITION_INDEX = 2  # ACTR-avi-Cy3B_NCBD-CF660R_50mMNaPi (base condition)

In [ ]:
condition_name = conditions[CONDITION_INDEX]
condition_dir = DATA_ROOT / condition_name
traj_files = sorted([str(f) for f in condition_dir.glob("*.h5")])
print(f"Condition: {condition_name}")
print(f"Number of trajectories: {len(traj_files)}")

# Load CSV metadata for cross-reference
df_meta = pd.read_csv(CSV_PATH)
df_condition = df_meta[df_meta["path"].str.contains(condition_name)]
print(f"CSV entries for this condition: {len(df_condition)}")

In [ ]:
# Load all trajectory data
durations_ms = []
n_photons_donor = []
n_photons_acceptor = []
all_interphoton = []
all_detectors = []
all_macro_units = []

for traj_file in traj_files:
    dur, ipt, mtu, det = get_trajectory_data(traj_file)
    durations_ms.append(dur)
    n_photons_donor.append(np.sum(det == DONOR_ID))
    n_photons_acceptor.append(np.sum(det == ACCEPTOR_ID))
    all_interphoton.append(ipt)
    all_detectors.append(det)
    all_macro_units.append(mtu)

durations_ms = np.array(durations_ms)
n_photons_donor = np.array(n_photons_donor)
n_photons_acceptor = np.array(n_photons_acceptor)
n_photons_total = n_photons_donor + n_photons_acceptor

print(f"Loaded {len(durations_ms)} trajectories")

### Summary Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(durations_ms, bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Duration (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Trajectory Durations")
axes[0].axvline(np.median(durations_ms), color="red", ls="--", label=f"median={np.median(durations_ms):.1f} ms")
axes[0].legend()

axes[1].hist(n_photons_total, bins=30, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Total Photons")
axes[1].set_ylabel("Count")
axes[1].set_title("Photon Counts")
axes[1].axvline(np.median(n_photons_total), color="red", ls="--", label=f"median={np.median(n_photons_total):.0f}")
axes[1].legend()

rates = n_photons_total / durations_ms  # photons/ms
sc = axes[2].scatter(durations_ms, n_photons_total, c=rates, cmap="viridis", alpha=0.6, s=15)
axes[2].set_xlabel("Duration (ms)")
axes[2].set_ylabel("Total Photons")
axes[2].set_title("Photons vs Duration")
plt.colorbar(sc, ax=axes[2], label="Rate (photons/ms)")

plt.tight_layout()
plt.show()

print(f"Duration  — mean: {durations_ms.mean():.1f} ms, median: {np.median(durations_ms):.1f} ms, std: {durations_ms.std():.1f} ms")
print(f"Photons   — mean: {n_photons_total.mean():.0f}, median: {np.median(n_photons_total):.0f}, std: {n_photons_total.std():.0f}")
print(f"Rate      — mean: {rates.mean():.1f} photons/ms, median: {np.median(rates):.1f} photons/ms")

### Example Traces

In [ ]:
BIN_WIDTH_MS = 0.25      # coarse bin width for visualization (ms)
SIM_BIN_WIDTH_MS = 0.005  # fine bin width for simulator output (ms, e.g. 5 us)

# Pick 4 evenly spaced trajectories
example_indices = np.linspace(0, len(traj_files) - 1, 4, dtype=int)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, idx in zip(axes.flat, example_indices):
    t, donor, acceptor = photons_to_timeseries(
        all_interphoton[idx], all_macro_units[idx], all_detectors[idx], bin_width_ms=BIN_WIDTH_MS
    )
    ax.plot(t, donor, color="green", alpha=0.8, label="Donor")
    ax.plot(t, acceptor, color="red", alpha=0.8, label="Acceptor")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Photon Count")
    ax.set_title(f"Traj {idx}: {durations_ms[idx]:.1f} ms, {n_photons_total[idx]} photons")
    ax.legend(fontsize=8)

plt.suptitle(f"Example traces (bin width = {BIN_WIDTH_MS} ms)", fontsize=14)
plt.tight_layout()
plt.show()

# Section 2: Extract Key Observables

We extract photon rates, FRET efficiencies, and timing properties from the data to guide simulator parameter choices.

### 2.1 Photon Rates

The total photon rate constrains **kD** (donor excitation rate). In the simulator, the expected photon count per time bin is approximately:

$$\text{count/bin} \approx \eta \cdot (k_D \cdot \phi + k_{bg}) \cdot dt \cdot \text{saving\_freq}$$

where $\phi$ is the distance-dependent emission probability and $\eta$ is detector efficiency.

In [ ]:
donor_rates = n_photons_donor / durations_ms      # photons/ms
acceptor_rates = n_photons_acceptor / durations_ms
total_rates = n_photons_total / durations_ms

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, label, color in zip(
    axes,
    [donor_rates, acceptor_rates, total_rates],
    ["Donor Rate", "Acceptor Rate", "Total Rate"],
    ["green", "red", "steelblue"],
):
    ax.hist(data, bins=30, edgecolor="black", alpha=0.7, color=color)
    ax.axvline(np.median(data), color="black", ls="--", label=f"median={np.median(data):.2f}")
    ax.set_xlabel(f"{label} (photons/ms)")
    ax.set_ylabel("Count")
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

print(f"Median total photon rate: {np.median(total_rates):.2f} photons/ms")
print(f"Median donor rate:        {np.median(donor_rates):.2f} photons/ms")
print(f"Median acceptor rate:     {np.median(acceptor_rates):.2f} photons/ms")

### 2.2 FRET Efficiency Distribution

Per-trajectory FRET efficiency: $E = n_{acceptor} / (n_{acceptor} + n_{donor})$

In [ ]:
# Per-trajectory FRET efficiency
E_traj = n_photons_acceptor / (n_photons_acceptor + n_photons_donor)

# Time-resolved E in bins (pooled across all trajectories)
E_binned_all = []
for i in range(len(traj_files)):
    _, donor, acceptor = photons_to_timeseries(
        all_interphoton[i], all_macro_units[i], all_detectors[i], bin_width_ms=BIN_WIDTH_MS
    )
    total = donor + acceptor
    valid = total > 0
    if valid.any():
        E_binned_all.append(acceptor[valid] / total[valid])

E_binned_pooled = np.concatenate(E_binned_all)

E_5, E_50, E_95 = np.percentile(E_traj, [5, 50, 95])
E_min_obs = float(E_binned_pooled.min())
E_max_obs = float(E_binned_pooled.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(E_traj, bins=30, edgecolor="black", alpha=0.7, color="purple")
axes[0].axvline(E_50, color="black", ls="--", label=f"median={E_50:.2f}")
axes[0].axvline(E_5, color="gray", ls=":", label=f"5th={E_5:.2f}")
axes[0].axvline(E_95, color="gray", ls=":", label=f"95th={E_95:.2f}")
axes[0].set_xlabel("FRET Efficiency E")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-Trajectory FRET Efficiency")
axes[0].legend()

axes[1].hist(E_binned_pooled, bins=50, edgecolor="black", alpha=0.7, color="orchid")
axes[1].set_xlabel("FRET Efficiency E")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Binned FRET Efficiency (bin={BIN_WIDTH_MS} ms, all trajectories pooled)")

plt.tight_layout()
plt.show()

print(f"Per-trajectory E: median={E_50:.3f}, 5th={E_5:.3f}, 95th={E_95:.3f}")
print(f"Binned E range:   min={E_min_obs:.3f}, max={E_max_obs:.3f}")

### 2.3 FRET Efficiency to Distance Mapping

The FRET efficiency depends on inter-dye distance $r$ and the Forster radius $R_0$:

$$E = \frac{1}{1 + (r / R_0)^6}$$

Inverting: $r = R_0 \cdot \left(\frac{1-E}{E}\right)^{1/6}$

This maps the observed E range to a physical distance range, which constrains **x_min** and **x_max**.

In [ ]:
# === Forster radius for your dye pair ===
R0_LITERATURE = 5.4  # nm, Cy3B / CF660R

# Convert FRET efficiency to distance: r = R0 * ((1-E)/E)^(1/6)
def E_to_distance(E, R0):
    E = np.clip(E, 1e-6, 1 - 1e-6)  # avoid division by zero
    return R0 * ((1 - E) / E) ** (1 / 6)

# Distance range from percentiles (active region)
r_from_E95 = E_to_distance(E_95, R0_LITERATURE)  # high E -> short distance
r_from_E5 = E_to_distance(E_5, R0_LITERATURE)    # low E -> long distance

# Distance range from full observed E (min/max from binned data)
r_from_Emax = E_to_distance(E_max_obs, R0_LITERATURE)  # highest E -> shortest r
r_from_Emin = E_to_distance(E_min_obs, R0_LITERATURE)  # lowest E -> longest r

# Plot E vs r curve
r_plot = np.linspace(2.0, 10.0, 500)
E_plot = 1 / (1 + (r_plot / R0_LITERATURE) ** 6)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_plot, E_plot, "k-", lw=2)

# Full observed range (min/max)
ax.axhline(E_min_obs, color="coral", ls="-", alpha=0.5, label=f"E min = {E_min_obs:.2f}")
ax.axhline(E_max_obs, color="coral", ls="-", alpha=0.5, label=f"E max = {E_max_obs:.2f}")
ax.axvline(r_from_Emin, color="coral", ls="-", alpha=0.5, label=f"r(E_min) = {r_from_Emin:.1f} nm")
ax.axvline(r_from_Emax, color="coral", ls="-", alpha=0.5, label=f"r(E_max) = {r_from_Emax:.1f} nm")
ax.fill_betweenx([0, 1], r_from_Emax, r_from_Emin, alpha=0.10, color="coral")

# Percentile range (5th/95th)
ax.axhline(E_5, color="gray", ls=":", alpha=0.5, label=f"E 5th = {E_5:.2f}")
ax.axhline(E_95, color="gray", ls=":", alpha=0.5, label=f"E 95th = {E_95:.2f}")
ax.axvline(r_from_E5, color="steelblue", ls="--", alpha=0.5)
ax.axvline(r_from_E95, color="steelblue", ls="--", alpha=0.5)
ax.fill_betweenx([0, 1], r_from_E95, r_from_E5, alpha=0.15, color="steelblue", label="5th-95th pctl")

ax.set_xlabel("Distance r (nm)", fontsize=14)
ax.set_ylabel("FRET Efficiency E", fontsize=14)
ax.set_title(f"FRET Efficiency vs Distance (R0 = {R0_LITERATURE} nm)")
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

margin = 2  # nm
print(f"Full observed E range [{E_min_obs:.2f}, {E_max_obs:.2f}] at R0 = {R0_LITERATURE} nm")
print(f"  -> full distance range [{r_from_Emax:.1f}, {r_from_Emin:.1f}] nm")
print(f"Suggested domain (+ {margin} nm margin):")
print(f"  x_min ~ {r_from_Emax + margin:.1f} nm,  x_max ~ {r_from_Emin - margin:.1f} nm")

### 2.4 Trajectory Timing

In [ ]:
# Number of time bins per trajectory at visualization and simulator bin widths
n_bins_per_traj = np.array([max(1, int(np.ceil(d / BIN_WIDTH_MS))) for d in durations_ms])
n_bins_sim_res = np.array([max(1, int(np.ceil(d / SIM_BIN_WIDTH_MS))) for d in durations_ms])

min_bins = int(n_bins_per_traj.min())
p50_bins = int(np.percentile(n_bins_per_traj, 50))
p95_bins = int(np.percentile(n_bins_per_traj, 95))
max_bins_obs = int(n_bins_per_traj.max())
min_bins_obs = int(n_bins_per_traj.min())

p50_sim = int(np.percentile(n_bins_sim_res, 50))
p95_sim = int(np.percentile(n_bins_sim_res, 95))
max_sim = int(n_bins_sim_res.max())
min_sim = int(n_bins_sim_res.min())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(n_bins_per_traj, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(min_bins, color="purple", ls="--", label=f"min = {min_bins}")
axes[0].axvline(p50_bins, color="red", ls="--", label=f"median = {p50_bins}")
axes[0].axvline(p95_bins, color="orange", ls="--", label=f"95th = {p95_bins}")
axes[0].set_xlabel(f"Number of Bins (bin width = {BIN_WIDTH_MS} ms)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Visualization bins ({BIN_WIDTH_MS} ms)")
axes[0].legend()

axes[1].hist(n_bins_sim_res, bins=30, edgecolor="black", alpha=0.7, color="teal")
axes[1].axvline(min_sim, color="purple", ls="--", label=f"min = {min_sim:,}")
axes[1].axvline(p50_sim, color="red", ls="--", label=f"median = {p50_sim:,}")
axes[1].axvline(p95_sim, color="orange", ls="--", label=f"95th = {p95_sim:,}")
axes[1].set_xlabel(f"Number of Bins (bin width = {SIM_BIN_WIDTH_MS} ms)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Simulator bins ({SIM_BIN_WIDTH_MS} ms)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"At BIN_WIDTH_MS = {BIN_WIDTH_MS} ms (visualization):")
print(f"  Minimum bins/trajectory: {min_bins}")
print(f"  Median bins/trajectory:  {p50_bins}")
print(f"  95th percentile:         {p95_bins}")
print(f"  Maximum:                 {max_bins_obs}")
print(f"  Minimum:                 {min_bins_obs:,}")
print()
print(f"At SIM_BIN_WIDTH_MS = {SIM_BIN_WIDTH_MS} ms (simulator resolution):")
print(f"  Minimum bins/trajectory: {min_sim:,}")
print(f"  Median bins/trajectory:  {p50_sim:,}")
print(f"  95th percentile:         {p95_sim:,}")
print(f"  Maximum:                 {max_sim:,}")
print(f"  Minimum:                 {min_sim:,}")
print(f"  -> max_bins must accommodate these counts for binding mode")

In [ ]:
# Collect all observables
observables = {
    "median_total_rate_per_ms": float(np.median(total_rates)),
    "median_donor_rate_per_ms": float(np.median(donor_rates)),
    "median_acceptor_rate_per_ms": float(np.median(acceptor_rates)),
    "median_E": float(E_50),
    "E_5th": float(E_5),
    "E_95th": float(E_95),
    "E_min": float(E_min_obs),
    "E_max": float(E_max_obs),
    "r_min_nm": float(r_from_Emax),
    "r_max_nm": float(r_from_Emin),
    "median_duration_ms": float(np.median(durations_ms)),
    "p50_n_bins": p50_bins,
    "p95_n_bins": p95_bins,
    "max_n_bins": max_bins_obs,
    "p50_n_bins_sim": p50_sim,
    "p95_n_bins_sim": p95_sim,
    "max_n_bins_sim": max_sim,
}

print("=" * 50)
print("       OBSERVABLE SUMMARY")
print("=" * 50)
for k, v in observables.items():
    if isinstance(v, float):
        print(f"  {k:30s}  {v:.3f}")
    else:
        print(f"  {k:30s}  {v}")
print("=" * 50)

# Section 3: Map Observables to Simulator Parameters

We now translate the extracted observables into concrete prior bounds and fixed simulator settings. Each subsection has a **USER CONFIGURATION** block at the top — adjust these values based on the observables above and your domain knowledge.

In [ ]:
# === USER CONFIGURATION: Config Source ===
# If CONFIG_PATH is None, the notebook builds the config below from cells 3.1-3.4
# (hand-built). Set CONFIG_PATH to a YAML file under conf/simulator/ (or any
# OmegaConf-compatible yaml with `prior:` + `simulator:` sections) to load that
# instead. This lets you rerun Section 4 against the same real-data condition to
# compare prior statistics across configs without editing cells 3.1-3.4.
CONFIG_PATH = "/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/conf/simulator/exp_6.yaml"            # e.g. "conf/simulator/exp_4.yaml"
CONFIG_TAG  = "handbuilt"         # short label shown in the diagnostic table

### 3.1 Photophysics Parameters

In [ ]:
# === USER CONFIGURATION: Photophysics ===

# R0: Forster radius (nm). Literature value for Cy3B/CF660R ~ 5.4 nm
R0_min, R0_max = 4.5, 6.5

# kD: Donor excitation rate (photons / (dt * stepper_call))
# Observed total rate ~ median_total_rate photons/ms
# count/bin = eta * (kD * phi + k_bg) * dt * saving_freq
# With eta~0.85, phi~0.5 (at E~0.5), k_bg~0:
#   kD ~ count_per_bin / (eta * phi * dt * saving_freq)
kD_min, kD_max = 25.0, 500.0

# Background photon rates
k_gb_min, k_gb_max = 0.0, 1.5
k_rb_min, k_rb_max = 0.0, 1.5

# Detector quantum efficiencies
eta_g_min, eta_g_max = 0.7, 1.0
eta_r_min, eta_r_max = 0.7, 1.0

# Spectral leak fractions. Photon conservation on each emission channel fixes
# the diagonals, so the simulator derives C_gg = 1 - C_gr and C_rr = 1 - C_rg.
C_gr_min, C_gr_max = 0.0, 0.1   # P(green photon -> red detector)
C_rg_min, C_rg_max = 0.0, 0.1   # P(red photon -> green detector)

# --- Calibration rationale for kD ---
print("kD calibration estimate:")
print(f"  Observed median total rate: {observables['median_total_rate_per_ms']:.2f} photons/ms")
print(f"  With dt=5e-5 ms, saving_freq=10000 -> bin_width=0.5 ms")
print(f"  Expected count/bin at median rate: {observables['median_total_rate_per_ms'] * BIN_WIDTH_MS:.1f} photons")
print(f"  kD range [{kD_min}, {kD_max}] should produce comparable count rates")


### 3.2 Spatial Domain and Free Energy Landscape

In [ ]:
# === USER CONFIGURATION: Spatial Domain ===

# x_min, x_max: simulation domain in nm
# Should cover the distance range from E mapping, plus margin for boundary knots
x_min = 4.0
x_max = 7.0

# Force landscape spline configuration
num_knots = 7

# center: mean force values at each interior knot
# For a two-state potential (left well = bound, right well = unbound),
# positive force on left pushes right, negative on right pushes left
center = [-5.0, -3.0, 1.0, 0.0 ,1.0, 3.0, 5.0]

# scale: half-width of uniform prior around each center
# scale=0 -> fixed forces (not inferred), only logD + photophysics vary
# scale>0 -> force knots become part of the prior
scale = 4.5

# Spline interpolation and boundary settings
spline_type = "steffen"         # monotonicity-preserving
boundary_condition = "linear"   # linear extrapolation at domain edges
num_boundary_knots = 5
max_force = 40.0                # pN, boundary force amplitude
x_init = [3.0, 8.0]            # initial position range (uniform)

print(f"Domain: [{x_min}, {x_max}] nm")
print(f"Knots: {num_knots}, center={center}, scale={scale}")
print(f"  -> Force knots are {'FIXED' if scale == 0 else 'INFERRED (part of prior)'}")
print(f"Observed distance range: [{observables['r_min_nm']:.1f}, {observables['r_max_nm']:.1f}] nm")

### 3.3 Dynamics: logD

In [ ]:
# === USER CONFIGURATION: Diffusion ===

# logD = log10(D), where D is the diffusion coefficient in nm^2/ms.
# All times in the config are in milliseconds (ms). kBT = 1 in simulator units.
# Langevin equation: dx = D*F*dt + sqrt(2*D*dt)*noise
# For intrinsically disordered proteins: D ~ 0.3 to 3 nm^2/ms -> logD in [-0.5, 0.5]
logD_min, logD_max = 0.0, 1.0

print(f"logD range: [{logD_min}, {logD_max}]")
print(f"  -> D range: [{10**logD_min:.2f}, {10**logD_max:.2f}] nm^2/ms")

### 3.4 Timing Parameters

In [ ]:
# === USER CONFIGURATION: Timing ===
# All times in milliseconds (ms). D in nm^2/ms. kBT = 1.

time_step = 5e-5  # ms (integration dt = 50 ns)

# SIM_BIN_WIDTH_MS is defined in Cell 11 (currently {SIM_BIN_WIDTH_MS} ms).
# saving_frequency = number of integration steps per output bin.
saving_frequency = int(SIM_BIN_WIDTH_MS / time_step)

# Equilibrium mode: total simulated duration
total_time = 10000.0  # ms

# Binding mode: two-way shooting from x_TS, variable-length traces.
# L_max_bins = structural upper bound (window centered on the transition).
# L_min_bins = lower bound; each accepted trace has length L_j ~ U[L_min, L_max].
use_binding_mode = True
L_max_bins = int(p95_sim)
L_min_bins = int(min_sim)
k_persist = 150             # consecutive in-basin bins to declare commitment
landmark_grid_n = 1000      # grid size for locating x_A, x_B, x_TS

# Derived quantities
n_steps = int(total_time / time_step)
n_bins_eq = int(n_steps / saving_frequency)
actual_bin_width = saving_frequency * time_step  # should equal SIM_BIN_WIDTH_MS

# How many simulator bins per visualization bin (for rebinning plots)
sim_bins_per_vis_bin = max(1, int(round(BIN_WIDTH_MS / SIM_BIN_WIDTH_MS)))

print(f"time_step:             {time_step} ms")
print(f"saving_frequency:      {saving_frequency}")
print(f"SIM_BIN_WIDTH_MS:      {actual_bin_width} ms  (simulator native bin)")
print(f"BIN_WIDTH_MS:          {BIN_WIDTH_MS} ms  (visualization bin)")
print(f"sim bins per vis bin:  {sim_bins_per_vis_bin}")
print()
print(f"Equilibrium mode:")
print(f"  total_time:          {total_time} ms")
print(f"  total steps:         {n_steps:,}")
print(f"  output bins:         {n_bins_eq:,}")
print()
print(f"Binding mode (two-way shooting from x_TS):")
print(f"  L_min_bins:          {L_min_bins} sim bins  ({L_min_bins * actual_bin_width:.1f} ms)")
print(f"  L_max_bins:          {L_max_bins} sim bins  ({L_max_bins * actual_bin_width:.1f} ms)")
print(f"  k_persist:           {k_persist}")
print(f"  landmark_grid_n:     {landmark_grid_n}")


### 3.5 Build Configuration

In [ ]:
if CONFIG_PATH is not None:
    _cfg_path = Path(CONFIG_PATH)
    if not _cfg_path.is_absolute():
        _cfg_path = (project_root / _cfg_path).resolve()
    print(f"Loading config from: {_cfg_path}")
    config = OmegaConf.load(_cfg_path)
    if CONFIG_TAG == "handbuilt":
        CONFIG_TAG = _cfg_path.stem
else:
    print("Using hand-built config from cells 3.1-3.4")
    config = OmegaConf.create({
        "prior": {
            "logD_lims": [logD_min, logD_max],
            "R0_lims": [R0_min, R0_max],
            "kD_lims": [kD_min, kD_max],
            "k_gb_lims": [k_gb_min, k_gb_max],
            "k_rb_lims": [k_rb_min, k_rb_max],
            "eta_g_lims": [eta_g_min, eta_g_max],
            "eta_r_lims": [eta_r_min, eta_r_max],
            "C_gr_lims": [C_gr_min, C_gr_max],
            "C_rg_lims": [C_rg_min, C_rg_max],
            "num_knots": num_knots,
            "center": center,
            "scale": scale,
        },
        "simulator": {
            "mode": "binding" if use_binding_mode else "equilibrium",
            "total_time": total_time,
            "saving_frequency": saving_frequency,
            "time_step": time_step,
            "spline_type": spline_type,
            "boundary_condition": boundary_condition,
            "num_boundary_knots": num_boundary_knots,
            "max_force": max_force,
            "x_min": x_min,
            "x_max": x_max,
            "x_init": list(x_init),
            "binding": {
                "L_min_bins": L_min_bins,
                "L_max_bins": L_max_bins,
                "k_persist": k_persist,
                "landmark_grid_n": landmark_grid_n,
                "N_max": 200000000,
            },
        },
    })

# Re-derive the plain-Python names used by cells 34+ from whichever config won.
x_min                = float(config.simulator.x_min)
x_max                = float(config.simulator.x_max)
num_knots            = int(config.prior.num_knots)
num_boundary_knots   = int(config.simulator.num_boundary_knots)
time_step            = float(config.simulator.time_step)
saving_frequency     = int(config.simulator.saving_frequency)
actual_bin_width     = time_step * saving_frequency                       # ms per sim bin
sim_bins_per_vis_bin = max(1, int(round(BIN_WIDTH_MS / actual_bin_width)))
use_binding_mode     = (config.simulator.get("mode", "equilibrium") == "binding")

print(f"Config tag: {CONFIG_TAG}")
print(OmegaConf.to_yaml(config))

# Section 4: Validate Prior and Compare Simulations to Data

Sample from the constructed prior, visualize free energy landscapes, and compare simulated traces against real data.

In [ ]:
prior = generate_fret_prior_from_config(config, device="cpu")
eval_free_energy = eval_free_energy_from_config(config)
eval_force = eval_force_from_config(config)

test_sample = prior.sample((1,))
print(f"Prior dimensionality: {test_sample.shape[1]}")
print(f"  = 1 (logD) + {num_knots} (force knots) + 10 (photophysics)")
print(f"Sample: {test_sample[0].numpy()}")

### 4.1 Prior Free Energy Landscapes

In [ ]:
N_PRIOR_DRAWS = 5
samples = prior.sample((N_PRIOR_DRAWS,))

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

# Extend plot range beyond [x_min, x_max] to show the confining boundary forces.
# Spline knots extend by num_boundary_knots * knot_spacing on each side.
knot_spacing = (x_max - x_min) / (num_knots - 1)
x_plot_margin = 0.6 * num_boundary_knots * knot_spacing
x_eval = np.linspace(x_min - x_plot_margin, x_max + x_plot_margin, 500)

for i in range(N_PRIOR_DRAWS):
    _, U = eval_free_energy(params=samples[i], x_eval=x_eval)
    ax1.plot(x_eval, U - U.min(), color="gray", alpha=0.15)

# Mark the flexible-knot domain
ax1.axvline(x_min, color="black", ls=":", alpha=0.5, label=f"x_min = {x_min}")
ax1.axvline(x_max, color="black", ls=":", alpha=0.5, label=f"x_max = {x_max}")

# Overlay FRET efficiency curve
E_curve = 1 / (1 + (x_eval / R0_LITERATURE) ** 6)
ax2.plot(x_eval, E_curve, color="blue", ls="--", lw=2, alpha=0.7)

ax1.set_xlabel("Distance (nm)", fontsize=14)
ax1.set_ylabel(r"$\Delta G\ [k_BT]$", fontsize=14, color="gray")
ax2.set_ylabel(f"FRET Efficiency (R0={R0_LITERATURE} nm)", fontsize=14, color="blue")
ax1.tick_params(axis="y", labelcolor="gray")
ax2.tick_params(axis="y", labelcolor="blue")
ax1.set_ylim(0, 15)
ax2.set_ylim(0, 1)
ax1.set_title(f"{N_PRIOR_DRAWS} prior draws: Free Energy Landscape")
ax1.legend(fontsize=8, loc="upper left")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Simulated vs Real Traces

Run a few simulations and compare side-by-side with real experimental traces.

In [ ]:
N_COMPARE = 10

if use_binding_mode:
    simulator = binding_simulator_wrapper_from_config(config)
else:
    simulator = smFRET_simulator_wrapper_from_config(config)

# Generate simulated traces (at SIM_BIN_WIDTH_MS resolution)
sim_samples = prior.sample((N_COMPARE,))
sim_traces = []
sim_lengths = []
for i in range(N_COMPARE):
    result = simulator(sim_samples[i])
    attempts = 0
    while result is None and attempts < 20:
        sim_samples[i] = prior.sample((1,)).squeeze(0)
        result = simulator(sim_samples[i])
        attempts += 1
    if result is None:
        print(f"Warning: could not generate trace {i} after 20 attempts")
        continue
    if isinstance(result, tuple):
        trace, length = result
        sim_traces.append(trace[:length].numpy() if isinstance(trace, torch.Tensor) else trace[:length])
        sim_lengths.append(length)
    else:
        trace = result.numpy() if isinstance(result, torch.Tensor) else result
        sim_traces.append(trace)
        sim_lengths.append(len(trace))

print(f"Generated {len(sim_traces)} simulated traces (native bin = {SIM_BIN_WIDTH_MS} ms)")
for i, (tr, sl) in enumerate(zip(sim_traces, sim_lengths)):
    print(f"  Trace {i}: {sl} sim bins ({sl * actual_bin_width:.1f} ms), rebinned to {sl // sim_bins_per_vis_bin} vis bins")

In [ ]:
# Side-by-side comparison: Real (left) vs Simulated (right)
# Real traces are binned at BIN_WIDTH_MS, simulated traces are rebinned from SIM_BIN_WIDTH_MS -> BIN_WIDTH_MS
n_show = min(N_COMPARE, len(sim_traces))
real_indices = np.linspace(0, len(traj_files) - 1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 2, figsize=(14, 3.5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, 2)

for row in range(n_show):
    # Real trace (binned at BIN_WIDTH_MS)
    idx = real_indices[row]
    t_real, donor_real, acc_real = photons_to_timeseries(
        all_interphoton[idx], all_macro_units[idx], all_detectors[idx], bin_width_ms=BIN_WIDTH_MS
    )
    axes[row, 0].plot(t_real, donor_real, color="green", alpha=0.8, label="Donor")
    axes[row, 0].plot(t_real, acc_real, color="red", alpha=0.8, label="Acceptor")
    axes[row, 0].set_ylabel("Photon Count")
    axes[row, 0].set_title(f"Real #{idx}: {durations_ms[idx]:.1f} ms", fontsize=10)
    axes[row, 0].legend(fontsize=7)

    # Simulated trace (rebinned from SIM_BIN_WIDTH_MS -> BIN_WIDTH_MS)
    tr_rebinned = rebin_simulated_trace(sim_traces[row], sim_bins_per_vis_bin)
    t_sim = np.arange(len(tr_rebinned)) * BIN_WIDTH_MS
    axes[row, 1].plot(t_sim, tr_rebinned[:, 0], color="green", alpha=0.8, label="Donor")
    axes[row, 1].plot(t_sim, tr_rebinned[:, 1], color="red", alpha=0.8, label="Acceptor")
    axes[row, 1].set_ylabel("Photon Count")
    axes[row, 1].set_title(f"Simulated #{row}: {sim_lengths[row] * actual_bin_width:.1f} ms", fontsize=10)
    axes[row, 1].legend(fontsize=7)

axes[-1, 0].set_xlabel("Time (ms)")
axes[-1, 1].set_xlabel("Time (ms)")
axes[0, 0].set_title("REAL " + axes[0, 0].get_title(), fontweight="bold")
axes[0, 1].set_title("SIMULATED " + axes[0, 1].get_title(), fontweight="bold")
plt.suptitle(f"Both traces at {BIN_WIDTH_MS} ms bin width (sim rebinned from {SIM_BIN_WIDTH_MS} ms)", fontsize=12)
plt.tight_layout()
plt.show()

### 4.3 Photon Count Statistics Comparison

Compare distributions of photon rates and FRET efficiency between real and simulated data.

In [ ]:
import multiprocessing as mp

# Generate more simulated traces for statistics (multiprocessing)
N_SIM_STATS = 1000
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)
ATTEMPTS_PER_TASK = 32  # tune for throughput

def _init_worker():
    # Ensure different RNG streams per worker
    seed = (os.getpid() + int.from_bytes(os.urandom(2), "little")) % (2**32 - 1)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)

def _simulate_stats_batch(n_attempts):
    batch_stats = []
    batch_rejected = 0

    params_batch = prior.sample((n_attempts,))
    for i in range(n_attempts):
        result = simulator(params_batch[i])

        if result is None:
            batch_rejected += 1
            continue

        if isinstance(result, tuple):
            trace, length = result
            length = int(length)
            trace = trace[:length]
            trace = trace.numpy() if isinstance(trace, torch.Tensor) else trace
        else:
            trace = result.numpy() if isinstance(result, torch.Tensor) else result

        tr_rebinned = rebin_simulated_trace(trace, sim_bins_per_vis_bin)
        if len(tr_rebinned) == 0:
            batch_rejected += 1
            continue

        duration_sim_ms = len(tr_rebinned) * BIN_WIDTH_MS
        donor_total = float(tr_rebinned[:, 0].sum())
        acceptor_total = float(tr_rebinned[:, 1].sum())
        total = donor_total + acceptor_total

        if total > 0.0 and duration_sim_ms > 0.0:
            batch_stats.append((total / duration_sim_ms, acceptor_total / total, duration_sim_ms))
        else:
            batch_rejected += 1

    return batch_stats, batch_rejected

all_stats = []
n_rejected = 0

ctx = mp.get_context("fork")  # Linux/macOS notebook-friendly
with ctx.Pool(processes=N_WORKERS, initializer=_init_worker) as pool:
    while len(all_stats) < N_SIM_STATS:
        remaining = N_SIM_STATS - len(all_stats)

        valid_so_far = len(all_stats)
        attempts_so_far = valid_so_far + n_rejected
        accept_rate = (valid_so_far / attempts_so_far) if attempts_so_far > 0 else 0.5
        accept_rate = max(0.1, accept_rate)

        attempts_needed = int(np.ceil(remaining / accept_rate))
        n_tasks = max(N_WORKERS, int(np.ceil(attempts_needed / ATTEMPTS_PER_TASK)))
        n_tasks = min(n_tasks, 8 * N_WORKERS)

        for batch_stats, batch_rej in pool.map(_simulate_stats_batch, [ATTEMPTS_PER_TASK] * n_tasks):
            all_stats.extend(batch_stats)
            n_rejected += batch_rej
            if len(all_stats) >= N_SIM_STATS:
                break

all_stats = all_stats[:N_SIM_STATS]

sim_total_rates = np.array([x[0] for x in all_stats], dtype=float)
sim_E_values = np.array([x[1] for x in all_stats], dtype=float)
sim_durations_ms = np.array([x[2] for x in all_stats], dtype=float)

n_generated = len(all_stats)
rejection_rate = n_rejected / (n_generated + n_rejected) if (n_generated + n_rejected) > 0 else np.nan

print(f"Generated {n_generated} valid simulated traces for statistics")
print(f"Workers: {N_WORKERS}, attempts/task: {ATTEMPTS_PER_TASK}")
print(f"Rejection rate: {rejection_rate:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Photon rate comparison
bins_rate = np.linspace(
    min(total_rates.min(), sim_total_rates.min()),
    max(total_rates.max(), sim_total_rates.max()),
    30,
)
axes[0].hist(total_rates, bins=bins_rate, alpha=0.5, color="steelblue", label="Real", density=True, edgecolor="black")
axes[0].hist(sim_total_rates, bins=bins_rate, alpha=0.5, color="orange", label="Simulated", density=True, edgecolor="black")
axes[0].set_xlabel("Total Photon Rate (photons/ms)")
axes[0].set_ylabel("Density")
axes[0].set_title("Photon Rate: Real vs Simulated")
axes[0].legend()

# FRET efficiency comparison
bins_E = np.linspace(0, 1, 30)
axes[1].hist(E_traj, bins=bins_E, alpha=0.5, color="steelblue", label="Real", density=True, edgecolor="black")
axes[1].hist(sim_E_values, bins=bins_E, alpha=0.5, color="orange", label="Simulated", density=True, edgecolor="black")
axes[1].set_xlabel("FRET Efficiency E")
axes[1].set_ylabel("Density")
axes[1].set_title("FRET Efficiency: Real vs Simulated")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic summary
print(f"  Config: {CONFIG_TAG}")
print("=" * 55)
print("  REAL vs SIMULATED DIAGNOSTIC")
print("=" * 55)

real_rate_med = np.median(total_rates)
sim_rate_med = np.median(sim_total_rates)
rate_ratio = sim_rate_med / real_rate_med if real_rate_med > 0 else float("inf")

real_E_med = np.median(E_traj)
sim_E_med = np.median(sim_E_values)

real_dur_med = np.median(durations_ms)
sim_dur_med = np.median(sim_durations_ms)
dur_ratio = sim_dur_med / real_dur_med if real_dur_med > 0 else float("inf")

print(f"  {'Metric':30s} {'Real':>10s} {'Simulated':>10s} {'Status':>10s}")
print(f"  {'-'*62}")

rate_status = "OK" if 0.5 < rate_ratio < 2.0 else "WARNING"
print(f"  {'Median photon rate (ph/ms)':30s} {real_rate_med:10.2f} {sim_rate_med:10.2f} {rate_status:>10s}")

E_diff = abs(sim_E_med - real_E_med)
E_status = "OK" if E_diff < 0.15 else "WARNING"
print(f"  {'Median FRET efficiency':30s} {real_E_med:10.3f} {sim_E_med:10.3f} {E_status:>10s}")

dur_status = "OK" if 0.3 < dur_ratio < 3.0 else "WARNING"
print(f"  {'Median duration (ms)':30s} {real_dur_med:10.2f} {sim_dur_med:10.2f} {dur_status:>10s}")

print(f"  {'-'*62}")
if rate_status == "WARNING":
    print(f"  -> Photon rate mismatch (ratio={rate_ratio:.2f}). Consider adjusting kD bounds.")
if E_status == "WARNING":
    print(f"  -> FRET E mismatch (diff={E_diff:.3f}). Consider adjusting R0, x_min/x_max, or force knots.")
if dur_status == "WARNING":
    print(f"  -> Duration mismatch (ratio={dur_ratio:.2f}). Consider adjusting logD, force landscape, or domain.")
if rate_status == "OK" and E_status == "OK" and dur_status == "OK":
    print(f"  -> All checks passed. Configuration looks reasonable.")

### 4.4 Transition Path Time Comparison

Compare the distribution of transition path durations between experimental and simulated data.
This is a key observable for binding kinetics — the simulator should produce
transition paths with similar timescales to the experiment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: Overlaid histograms ---
bins_dur = np.linspace(
    min(np.array(durations_ms).min(), sim_durations_ms.min()),
    max(np.array(durations_ms).max(), sim_durations_ms.max()),
    40,
)
axes[0].hist(durations_ms, bins=bins_dur, alpha=0.5, edgecolor="black",
             label=f"Real (n={len(durations_ms)})", color="steelblue", density=True)
axes[0].hist(sim_durations_ms, bins=bins_dur, alpha=0.5, edgecolor="black",
             label=f"Simulated (n={len(sim_durations_ms)})", color="coral", density=True)
axes[0].axvline(np.median(durations_ms), color="steelblue", ls="--", lw=1.5,
                label=f"Real median = {np.median(durations_ms):.2f} ms")
axes[0].axvline(np.median(sim_durations_ms), color="coral", ls="--", lw=1.5,
                label=f"Sim median = {np.median(sim_durations_ms):.2f} ms")
axes[0].set_xlabel("Transition Path Duration (ms)")
axes[0].set_ylabel("Density")
axes[0].set_title("Duration Distribution")
axes[0].legend(fontsize=8)

# --- Right: Log-scale for tail comparison ---
axes[1].hist(durations_ms, bins=bins_dur, alpha=0.5, edgecolor="black",
             label="Real", color="steelblue", density=True)
axes[1].hist(sim_durations_ms, bins=bins_dur, alpha=0.5, edgecolor="black",
             label="Simulated", color="coral", density=True)
axes[1].set_yscale("log")
axes[1].set_xlabel("Transition Path Duration (ms)")
axes[1].set_ylabel("Density (log)")
axes[1].set_title("Duration Distribution (log scale)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Transition path durations:")
print(f"  {'':20s} {'Real':>10s} {'Simulated':>10s}")
print(f"  {'Median (ms)':20s} {np.median(durations_ms):10.2f} {np.median(sim_durations_ms):10.2f}")
print(f"  {'Mean (ms)':20s} {np.mean(durations_ms):10.2f} {np.mean(sim_durations_ms):10.2f}")
print(f"  {'Std (ms)':20s} {np.std(durations_ms):10.2f} {np.std(sim_durations_ms):10.2f}")
print(f"  {'5th pctl (ms)':20s} {np.percentile(durations_ms, 5):10.2f} {np.percentile(sim_durations_ms, 5):10.2f}")
print(f"  {'95th pctl (ms)':20s} {np.percentile(durations_ms, 95):10.2f} {np.percentile(sim_durations_ms, 95):10.2f}")

In [ ]:
# === Save statistics summary for cross-config comparison ===
# Writes a small JSON with the medians from the real-vs-simulated diagnostic.
# Run the notebook twice with different CONFIG_PATH / CONFIG_TAG settings and
# compare the files under smFRET_sbi/analysis/setup_prior_stats/.
import json as _json

_out_dir = project_root / "analysis" / "setup_prior_stats"
_out_dir.mkdir(parents=True, exist_ok=True)
_stats = {
    "config_tag":  CONFIG_TAG,
    "config_path": str(CONFIG_PATH) if CONFIG_PATH else None,
    "condition":   condition_name,
    "real": {
        "median_rate_per_ms": float(np.median(total_rates)),
        "median_E":           float(np.median(E_traj)),
        "median_dur_ms":      float(np.median(durations_ms)),
    },
    "sim": {
        "median_rate_per_ms": float(np.median(sim_total_rates)),
        "median_E":           float(np.median(sim_E_values)),
        "median_dur_ms":      float(np.median(sim_durations_ms)),
    },
}
_out_path = _out_dir / f"{CONFIG_TAG}__{condition_name}.json"
_out_path.write_text(_json.dumps(_stats, indent=2))
print("Saved stats summary to", _out_path)
